In [1]:
# ==============================================================================
# 🎓 AI English Tutor v8.0 (BLACKWELL EDITION - Ultra Fast)
# GPU: RTX PRO 6000 Blackwell | Модель: Qwen2.5-1.5B-Instruct (bfloat16)
# Без квантования — максимальная скорость!
# ==============================================================================

# 1. Установка (bitsandbytes больше НЕ нужен!)
!pip install -q transformers accelerate gradio

import gc
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, GenerationConfig

# 2. Очистка памяти
gc.collect()
torch.cuda.empty_cache()

print("🔍 Проверка GPU...")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ Найден GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print("⚡ У тебя мощнейшая видеокарта! Загружаем модель без сжатия для максимальной скорости.")
else:
    print("⚠️ GPU не найден!")

print("⏳ Загрузка модели в bfloat16 (быстрый режим)...")

# 3. Загрузка модели БЕЗ квантования — в bfloat16 (нативный формат Blackwell)
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,   # 🔥 Нативный формат для Blackwell — быстрее float16!
    device_map="auto",
    # quantization_config УБРАН — он тебе не нужен с такой видеокартой
)

# 4. Пайплайн с настройками для живого диалога
llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=250,       # Достаточно для полного ответа с вопросом
    temperature=0.75,         # Чуть креативности для живого диалога
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.15
)
print("✅ Модель готова к работе! Отвечает мгновенно. ⚡")

# ==============================================================================
# 5. ЛОГИКА: ЖИВОЙ ДИАЛОГ С ВОПРОСАМИ
# ==============================================================================

def get_tutor_response(message, history, topic, level):
    """AI ведет живой диалог: задает вопросы, предлагает слова, корректирует"""

    # НОВЫЙ ПРОМПТ: живой репетитор, который ведет диалог
    system_prompt = f"""You are a friendly and encouraging English tutor having a natural conversation with a student.
Student level: {level}
Current topic: {topic}

YOUR ROLE:
- Have a natural, engaging conversation about {topic}
- Ask interesting follow-up questions to keep the student talking
- If the student makes grammar mistakes, gently correct them and explain briefly in Russian
- Suggest 1-2 new vocabulary words related to {topic} that the student can try using in their next answer
- Keep responses SHORT and friendly (3-5 sentences max)
- ALWAYS end your message with a question to continue the conversation

RESPONSE FORMAT:
1. React naturally to student's answer (praise, show interest, or react to the content)
2. If there's a grammar mistake: show the correction + brief rule in Russian (e.g., "⚠️ Небольшая ошибка: 'I go' → 'I went' (Past Simple)")
3. Suggest a new word/phrase to try (with Russian translation in brackets)
4. Ask a follow-up question about {topic} to continue the dialogue

EXAMPLE OF GOOD RESPONSE:
"That's amazing! 🌟 So you've been to Italy — what was your favorite city there?
⚠️ Small note: 'I have went' → 'I have gone' (Past Participle).
💡 Try using the word 'memorable' (незабываемый) in your next answer. What was the most memorable moment of your trip?"

Keep it warm, use emojis occasionally, and make the student want to keep talking!"""

    messages = [{"role": "system", "content": system_prompt}]

    # Берем последние 8 сообщений для контекста
    for h in history[-8:]:
        messages.append({"role": h["role"], "content": h["content"]})

    messages.append({"role": "user", "content": message})

    # Применяем шаблон чата Qwen
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = llm_pipe(prompt)
    full_text = output[0]["generated_text"]

    # Извлекаем только новый ответ
    if "<|im_start|>assistant\n" in full_text:
        response = full_text.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    else:
        response = full_text[len(prompt):].strip()

    return response


def start_conversation(topic, level):
    """AI начинает диалог с приветствия и первого вопроса"""

    prompt = f"""You are starting an English conversation lesson with a student.
Topic: {topic}
Student level: {level}

Create a friendly opening message (3-5 sentences):
1. Warm greeting with emoji
2. Briefly introduce the topic and why it's interesting
3. Suggest 3 useful words/phrases for this topic (with Russian translations in brackets)
4. Ask an engaging opening question about {topic} to start the conversation

Be warm, encouraging, and use emojis. End with your question!"""

    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    output = llm_pipe(prompt_text)
    full_text = output[0]["generated_text"]

    if "<|im_start|>assistant\n" in full_text:
        response = full_text.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    else:
        response = full_text[len(prompt_text):].strip()

    return response


# ==============================================================================
# 6. ИНТЕРФЕЙС
# ==============================================================================

TOPICS = [
    "Travel ✈️", "Hobbies 🎨", "Food & Cooking 🍕", "Movies & Series 🎬",
    "Work & Career 💼", "Technology & AI 💻", "Sports ⚽", "Music 🎵",
    "Books 📚", "Family & Friends 👨‍👩‍👧", "Health & Fitness 🏃", "Fashion 👗"
]
LEVELS = ["A1 (Beginner)", "A2 (Elementary)", "B1 (Intermediate)", "B2 (Upper-Intermediate)", "C1 (Advanced)"]

custom_css = """
.main-header {
    background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%);
    color: white;
    padding: 1.5rem;
    border-radius: 12px;
    text-align: center;
    margin-bottom: 1rem;
    box-shadow: 0 4px 15px rgba(245, 87, 108, 0.3);
}
.gradio-container { max-width: 1100px !important; margin: auto; }
.sidebar { background: #fff5f7; padding: 1.5rem; border-radius: 12px; border: 2px solid #ffe0e6; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="purple"), css=custom_css) as demo:

    gr.HTML("""
    <div class="main-header">
        <h1 style="margin:0;">💬 AI English Conversation Tutor</h1>
        <p style="margin:0.5rem 0 0 0; opacity:0.95;">Powered by Qwen2.5-1.5B on RTX Blackwell ⚡ Живой диалог с проверкой грамматики</p>
    </div>
    """)

    with gr.Row():
        # Левая панель
        with gr.Column(scale=1, elem_classes="sidebar"):
            gr.Markdown("### ⚙️ Настройки")

            topic_dd = gr.Dropdown(
                choices=TOPICS,
                value="Travel ✈️",
                label="🎯 Тема разговора",
                info="Выбери интересную тему"
            )

            level_dd = gr.Dropdown(
                choices=LEVELS,
                value="B1 (Intermediate)",
                label="📊 Твой уровень",
                info="Выбери свой уровень"
            )

            start_btn = gr.Button("🚀 Начать разговор", variant="primary", size="lg")

            gr.Markdown("---")
            gr.Markdown("""
            ### 💡 Как это работает:
            1. Выбери тему и уровень
            2. AI начнет диалог и задаст первый вопрос
            3. Отвечай на английском
            4. AI будет:
               - Поддерживать беседу
               - Мягко исправлять ошибки
               - Предлагать новые слова
               - Задавать новые вопросы
            5. Просто болтай! 🗣️
            """)

        # Правая панель - чат
        with gr.Column(scale=3):
            gr.Markdown("### 💬 Диалог с репетитором")

            chatbot = gr.Chatbot(
                type="messages",
                height=600,
                show_label=False,
                avatar_images=(
                    "https://em-content.zobj.net/source/twitter/376/student_1f9d1.png",
                    "https://em-content.zobj.net/source/twitter/376/robot_1f916.png"
                ),
                render_markdown=True
            )

            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="✍️ Напиши ответ на английском...",
                    lines=3,
                    scale=4,
                    container=False
                )
                send_btn = gr.Button("📤", variant="primary", scale=1)

            clear_btn = gr.Button("🗑️ Очистить чат", variant="secondary", size="sm")

    # Обработчики
    def on_start(topic, level):
        response = start_conversation(topic, level)
        return [{"role": "assistant", "content": response}]

    def on_send(msg, history, topic, level):
        if not msg.strip():
            return history, ""

        new_history = history + [{"role": "user", "content": msg}]
        response = get_tutor_response(msg, history, topic, level)
        new_history.append({"role": "assistant", "content": response})

        return new_history, ""

    start_btn.click(
        fn=on_start,
        inputs=[topic_dd, level_dd],
        outputs=[chatbot]
    )

    send_btn.click(
        fn=on_send,
        inputs=[msg_input, chatbot, topic_dd, level_dd],
        outputs=[chatbot, msg_input]
    )

    msg_input.submit(
        fn=on_send,
        inputs=[msg_input, chatbot, topic_dd, level_dd],
        outputs=[chatbot, msg_input]
    )

    clear_btn.click(lambda: [], None, chatbot)

print("🌐 Запуск интерфейса... Ссылка появится ниже.")
demo.launch(share=True, debug=False)

🔍 Проверка GPU...
✅ Найден GPU: NVIDIA A100-SXM4-40GB (39.5 GB)
⚡ У тебя мощнейшая видеокарта! Загружаем модель без сжатия для максимальной скорости.
⏳ Загрузка модели в bfloat16 (быстрый режим)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'repetition_penalty', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_1480/1708893903.py:163: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="purple"), css=custom_css) as demo:
/tmp/ipykernel_1480/1708893903.py:163: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="pink", secondary_hue="purple"), css=custom_css) as demo:
/tmp/ipykernel_1480/1708893903.py:211: DeprecationWarning: The default value of 'allow_t

✅ Модель готова к работе! Отвечает мгновенно. ⚡
🌐 Запуск интерфейса... Ссылка появится ниже.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c9e5e25c8cbcca209c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
